# Multi-Label AMR Classifier — per-drug class weighting (04-04)

**Same as 04-00 but with `pos_weight` in `BCEWithLogitsLoss`, computed per drug
from the training labels.** Rare resistant classes contribute more to the loss
without resampling. Reference implementation for the documentation page
[docs/04](https://asd-standard.github.io/Dryad_DRIAMS/04-Multi-Label-mlp/index.html).

## Protocol
- **Cross-site** — train on A (species-stratified 70/30 train/val in this
  variant), test on B/C/D; preprocessing, grid and thresholds use A
  train/val only.
- **Aggregated** — the four sites pooled and split species-stratified
  70/15/15, with per-drug thresholds tuned on the aggregated validation split.
- **Model** — shared spectral MLP (512 → 256 → 128, no attention) with masked,
  class-weighted BCE; 6×6 lr × dropout grid; best-validation-loss checkpoint.
- **Baseline** — OneVsRest L2 LR with per-drug tuned thresholds.

## Outputs (in `results_multilabel/`, not versioned)
Per-drug CSVs, heatmaps, comparison figures and the joblib summary. Committed
with outputs stripped.

## Notes
- Class weighting rescales only the positive BCE term; decision thresholds are
  still tuned per drug on validation.
- The aggregated run used the legacy 26,642-sample cache (corrected union:
  26,781; see the docs provenance note).
- Single seed (42); metrics are point estimates.


In [ ]:
!pip install maldideepkit maldiamrkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from tqdm import tqdm

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score, accuracy_score)

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
    DATA_ROOT = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet/Processed/Processing/Analysis/04-03-Wider-Backbone-Ablation")
else:
    DRYAD = Path("/media/asd/8f69beed-e984-445f-b8b3-abbb6a1a4b3f/Dryad-DataSet")

OUT_DIR = Path("./results_multilabel")
OUT_DIR.mkdir(exist_ok=True)

BIN_COLS = [f"bin_{i}" for i in range(6000)]
DRUGS_10 = ["Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic acid",
            "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
            "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin"]
SITES = ["DRIAMS-A", "DRIAMS-B", "DRIAMS-C", "DRIAMS-D"]
print(f"Drugs: {len(DRUGS_10)}  |  Sites: {SITES}")

In [ ]:
# =============================================================================
# 1. MULTI-LABEL DATASET (handles NaN = untested drugs)
# =============================================================================
# Defines MultiLabelDataset (spectra + labels + label mask) and
# load_multilabel_split(), which builds the per-site union matrices from the
# per-drug CSVs - deduplicating by ``code`` and storing NaN for untested drugs.

class MultiLabelDataset(Dataset):
    """Torch dataset yielding ``(spectrum, labels, label_mask)``.

    ``labels`` has NaNs replaced by 0 (those entries are never used by the
    loss); ``mask`` is 1.0 where the drug was tested and 0.0 otherwise.
    """
    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(np.nan_to_num(Y, nan=0.0), dtype=torch.float32)
        self.mask = torch.tensor(~np.isnan(Y), dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.mask[idx]


def load_multilabel_split():
    """Build the multi-label matrices from the per-drug Processed CSVs.

    For each site, spectra from the ten per-drug files are deduplicated by
    ``code``: the first occurrence defines X and species, while a drug label
    seen again for the same code overwrites the previous value. NaN marks
    untested drugs.

    Returns
    -------
    train_X, train_Y, train_sp : ndarray
        DRIAMS-A union, shapes ``(n_A, 6000)``, ``(n_A, 10)`` and species.
    test_sites : dict
        ``{"DRIAMS-B": (X, Y, species), "DRIAMS-C": ..., "DRIAMS-D": ...}``.
    """
    train_X, train_Y, train_sp = None, None, None
    test_sites = {}

    drug_dirs = {d: d.replace(" ", "_") for d in DRUGS_10}  # spaces become underscores in directory names

    for site_name in SITES:
        proc_site = f"Proc_{site_name}"
        site_labels = {}   # {code: {drug: label}}
        X_samples = {}     # {code: X_vector}
        sp_samples = {}    # {code: species}
        site_drugs_ok = []

        for drug_name in DRUGS_10:
            p = DATA_ROOT / proc_site / drug_name.replace(" ", "_") / "data.csv"
            if not p.exists():
                continue
            df = pd.read_csv(p)
            site_drugs_ok.append(drug_name)
            for _, row in df.iterrows():
                code = row["code"]
                if code not in X_samples:
                    X_samples[code] = row[BIN_COLS].values.astype("float32")
                    sp_samples[code] = row["species"]
                site_labels.setdefault(code, {})[drug_name] = row["label"]

        if not site_drugs_ok:
            continue

        codes = list(X_samples.keys())
        X = np.array([X_samples[c] for c in codes], dtype="float32")
        species = np.array([sp_samples[c] for c in codes])

        # Build the label matrix (N, 10); NaN marks drugs not tested for that
        # spectrum and is excluded from the loss and from every metric.
        Y = np.full((len(codes), len(DRUGS_10)), np.nan, dtype=float)
        for i, drug in enumerate(DRUGS_10):
            for j, code in enumerate(codes):
                if drug in site_labels.get(code, {}):
                    Y[j, i] = site_labels[code][drug]

        n_lab = (~np.isnan(Y)).sum(axis=1).mean()
        print(f"  {site_name}: {len(codes)} samples, {len(site_drugs_ok)} drugs, avg {n_lab:.1f} labels/sample")

        if site_name == "DRIAMS-A":
            train_X, train_Y, train_sp = X, Y, species
        else:
            test_sites[site_name] = (X, Y, species)

    print(f"\nTrain (A): {train_X.shape[0]} samples, {len(np.unique(train_sp))} species")
    for s, (Xs, Ys, sps) in test_sites.items():
        print(f"  Test {s}: {Xs.shape[0]} samples, {len(np.unique(sps))} species")
    return train_X, train_Y, train_sp, test_sites

---
## Multi-Label MLP Class Definition

Same wrapper as 04-00 with a weighted loss; the class docstring explains the
per-drug ``pos_weight`` semantics.

In [ ]:
# =============================================================================
# 2. MULTI-LABEL MLP CLASSIFIER  (sklearn-compatible, PyTorch backend)
# =============================================================================
# sklearn-compatible MLP identical to 04-00 except for the loss: fit()
# computes a per-drug pos_weight from the training labels and uses it in
# BCEWithLogitsLoss, so under-represented resistant classes weigh more.

class MultiLabelMaldiMLP(BaseEstimator, ClassifierMixin):
    """Sklearn-compatible multi-label MLP with per-drug class weighting.

    Identical to 04-00's wrapper except that :meth:`fit` computes a per-drug
    ``pos_weight`` for ``BCEWithLogitsLoss`` from the training labels, so rare
    resistant classes contribute proportionally more to the loss.

    Parameters
    ----------
    hidden_dim : int, default=512
    head_dims : tuple, default=(256, 128)
    use_attention : bool, default=False
    dropout_high, dropout_low : float
    learning_rate : float, default=1e-3
    weight_decay : float, default=0.0
    batch_size : int, default=64
    epochs : int, default=50
        Maximum epochs; early stopping may stop sooner.
    early_stopping_patience : int, default=10
    warmup_epochs : int, default=0
        Linear LR warmup before cosine annealing.
    val_fraction : float, default=0.1
    random_state : int, default=42
    verbose : bool, default=False
    device : str, default="cpu"

    Attributes (set by :meth:`fit`)
    -------------------------------
    model_, n_labels_, input_dim_, history_, best_epoch_, best_val_loss_,
    epochs_trained_, is_fitted_

    Notes
    -----
    Masked BCE: untested drugs are excluded from the loss. See docs/04 for the
    class-weighting ablation results.
    """
    def __init__(self, hidden_dim=512, head_dims=(256, 128), use_attention=False,
                 dropout_high=0.3, dropout_low=0.2,
                 learning_rate=1e-3, weight_decay=0.0,
                 batch_size=64, epochs=50, early_stopping_patience=10,
                 warmup_epochs=0, val_fraction=0.1,
                 random_state=42, verbose=False, device="cpu"):
        self.hidden_dim = hidden_dim
        self.head_dims = head_dims
        self.use_attention = use_attention
        self.dropout_high = dropout_high
        self.dropout_low = dropout_low
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.epochs = epochs
        self.early_stopping_patience = early_stopping_patience
        self.warmup_epochs = warmup_epochs
        self.val_fraction = val_fraction
        self.random_state = random_state
        self.verbose = verbose
        self.device = device

    def _build_model(self, input_dim, n_classes):
        """Instantiate the spectral MLP with one output logit per drug."""
        return SpectralAttentionMLP(
            input_dim=input_dim, n_classes=n_classes,
            hidden_dim=self.hidden_dim, head_dims=self.head_dims,
            use_attention=self.use_attention,
            dropout_high=self.dropout_high, dropout_low=self.dropout_low)

    def fit(self, X, Y):
        """Train with masked, class-weighted BCE; keep the best-validation-loss checkpoint.

        ``pos_weight`` is derived per drug from the training labels
        (#negatives / #positives), giving rare resistant classes a larger gradient
        share without resampling.

        Parameters
        ----------
        X : ndarray (N, 6000)
        Y : ndarray (N, 10)

        Returns
        -------
        self
        """
        np.random.seed(self.random_state)
        torch.manual_seed(self.random_state)

        self.n_labels_ = Y.shape[1]
        self.input_dim_ = X.shape[1]

        # ── Per-drug pos_weight for BCEWithLogitsLoss ──
        # Computed from the training labels only (never val/test), then frozen for
        # the whole fit - no leakage.
        self.pos_weights_ = torch.ones(self.n_labels_, dtype=torch.float32, device=self.device)
        for di in range(self.n_labels_):
            valid = ~np.isnan(Y[:, di])
            n_pos = max((Y[valid, di] == 1.0).sum(), 1)
            n_neg = max((Y[valid, di] == 0.0).sum(), 1)
            self.pos_weights_[di] = n_neg / n_pos
        if self.verbose:
            print(f"  pos_weights: {[f'{pw:.2f}' for pw in self.pos_weights_.cpu().numpy()]}")

        self.model_ = self._build_model(self.input_dim_, self.n_labels_).to(self.device)

        ds = MultiLabelDataset(X, Y)
        n_val = max(1, int(len(ds) * self.val_fraction))
        n_tr = len(ds) - n_val
        train_ds, val_ds = random_split(ds, [n_tr, n_val],
            generator=torch.Generator().manual_seed(self.random_state))

        train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=self.batch_size * 2, shuffle=False)

        opt_cls = torch.optim.AdamW if self.weight_decay > 0 else torch.optim.Adam
        optimizer = opt_cls(self.model_.parameters(), lr=self.learning_rate,
                            weight_decay=self.weight_decay)
        warmup = max(0, self.warmup_epochs)
        t_max = max(1, self.epochs - warmup)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=t_max, eta_min=1e-6)
        criterion = nn.BCEWithLogitsLoss(reduction="none", pos_weight=self.pos_weights_)

        best_val_loss = float("inf")
        best_state = None
        patience_counter = 0

        self.history_ = {"train_loss": [], "val_loss": []}
        for epoch in range(self.epochs):
            self.model_.train()
            train_loss = 0.0
            for xb, yb, mb in train_loader:
                xb, yb, mb = xb.to(self.device), yb.to(self.device), mb.to(self.device)
                if epoch < warmup:
                    for pg in optimizer.param_groups:
                        pg["lr"] = self.learning_rate * (epoch + 1) / warmup
                optimizer.zero_grad()
                logits = self.model_(xb)
                loss = criterion(logits, yb)
                loss = (loss * mb).sum() / mb.sum()
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            if epoch >= warmup:
                scheduler.step()

            self.model_.eval()
            val_loss = 0.0
            with torch.no_grad():
                for xb, yb, mb in val_loader:
                    xb, yb, mb = xb.to(self.device), yb.to(self.device), mb.to(self.device)
                    logits = self.model_(xb)
                    loss = criterion(logits, yb)
                    loss = (loss * mb).sum() / mb.sum()
                    val_loss += loss.item()

            val_loss /= len(val_loader)
            train_loss /= len(train_loader)

            self.history_["train_loss"].append(train_loss)
            self.history_["val_loss"].append(val_loss)
            if self.verbose:
                print(f"  Epoch {epoch+1:3d}/{self.epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in self.model_.state_dict().items()}
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.early_stopping_patience:
                    if self.verbose:
                        print(f"  Early stopping at epoch {epoch+1}")
                    break

        if best_state is not None:
            self.model_.load_state_dict(best_state)
        self.model_.eval()
        self.is_fitted_ = True
        return self

    def predict_proba(self, X):
        """Per-drug resistance probabilities, shape ``(N, 10)``."""
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            logits = self.model_(X_t)
            proba = torch.sigmoid(logits).cpu().numpy()
        return proba

    def predict(self, X, thresholds=None):
        """0/1 predictions; per-drug ``thresholds`` override the default 0.5 cut."""
        proba = self.predict_proba(X)
        if thresholds is None:
            thresholds = [0.5] * self.n_labels_
        return (proba >= np.array(thresholds)).astype(int)

---
## Data Loading + Cross-Site Split

A split species-stratified (70/30 in this variant); B/C/D are frozen test
sites and A validation drives the grid and thresholds.

In [ ]:
import joblib
# =============================================================================
# 3. LOAD DATA + CROSS-SITE SPLIT (train A, test B/C/D) - CACHED
# =============================================================================
# Loads the per-drug CSVs, builds the A union, splits it species-stratified
# 70/30 into train/val, fits log1p+standardization on the training split only
# and adds the pooled B+C+D test union.

cache_path = DATA_ROOT / "cross_site_multilabel_data.joblib"

if cache_path.exists():
    print(f"Loading cross-site data from cache: {cache_path}")
    X_train_raw, Y_train_raw, sp_train_raw, test_data_raw = joblib.load(cache_path)
else:
    print("Loading data from Processed CSVs (first time)...")
    X_train_raw, Y_train_raw, sp_train_raw, test_data_raw = load_multilabel_split()
    joblib.dump((X_train_raw, Y_train_raw, sp_train_raw, test_data_raw), cache_path)
    print(f"Data saved to cache: {cache_path}")

# Species-stratified A split (70/30 in this variant): the grid search and the
# per-drug thresholds use the validation part; B/C/D stay untouched test sites.
# stratified_species_drug_split handles rare species via min_count=2
idx_tr, idx_v, _, _ = stratified_species_drug_split(
    np.arange(len(Y_train_raw)).reshape(-1, 1),
    np.zeros(len(Y_train_raw)),  # dummy labels — stratification is by species only
    species=sp_train_raw,
    test_size=0.30,
    random_state=SEED)
idx_tr = idx_tr.flatten().astype(int)
idx_v = idx_v.flatten().astype(int)

X_train = X_train_raw[idx_tr]
Y_train = Y_train_raw[idx_tr]
X_val = X_train_raw[idx_v]
Y_val = Y_train_raw[idx_v]
print(f"\nSplit: A train={len(X_train)}  A val={len(X_val)}")

# log1p+standardization fitted on A train only, then applied to val and test.
state = fit_input_transform(X_train, "log1p+standardize")
X_train_pp = apply_input_transform(X_train, state)
X_val_pp = apply_input_transform(X_val, state)
print("Preprocessing done.")

# Apply preprocessing to test sites
test_pp = {}
for site, (Xs, Ys, _) in test_data_raw.items():
    test_pp[site] = (apply_input_transform(Xs, state), Ys)
    n_lab = (~np.isnan(Ys)).sum(axis=1).mean()
    print(f"  Test {site}: {Xs.shape[0]} samples preprocessed, avg {n_lab:.1f} labels/sample")

# Pooled B+C+D union row for the summary tables (weighted by site size).
if test_pp:
    X_combined = np.concatenate([t[0] for t in test_pp.values()])
    Y_combined = np.concatenate([t[1] for t in test_pp.values()])
    test_pp["B+C+D"] = (X_combined, Y_combined)
    print(f"  Test B+C+D combined: {X_combined.shape[0]} samples")

---
## Baseline: OneVsRest Logistic Regression

Linear reference under the identical split and threshold protocol.

In [ ]:
# =============================================================================
# 4. BASELINE: OneVsRest Logistic Regression
# =============================================================================
# Ten independent L2 logistic regressions (OneVsRest): C chosen by 3-fold CV
# on the training split, per-drug threshold tuned on A validation and frozen
# for the B/C/D evaluation.

C_grid = np.linspace(5e-5, 1e-3, 15)
thresholds = np.linspace(0.05, 0.95, 91)

lr_models = {}
lr_results = {}
for di, drug in enumerate(tqdm(DRUGS_10, desc="LR per drug")):
    # Training samples labeled for this drug; unlabeled samples are excluded.
    tr_mask = ~np.isnan(Y_train[:, di])
    X_tr, y_tr = X_train_pp[tr_mask], Y_train[tr_mask, di].astype(int)
    val_mask = ~np.isnan(Y_val[:, di])
    X_v, y_v = X_val_pp[val_mask], Y_val[val_mask, di].astype(int)

    if len(np.unique(y_tr)) < 2 or len(np.unique(y_v)) < 2:
        lr_results[drug] = {"A-val_BalAcc": np.nan, "A-val_AUC": np.nan}
        continue

    # 3-fold CV over the 15-value C grid on the training split.
    grid = GridSearchCV(
        LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000, random_state=SEED),
        param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=0)
    grid.fit(X_tr, y_tr)
    lr = grid.best_estimator_

    # Per-drug threshold tuned on A validation (91 values), frozen for test.
    proba_v = lr.predict_proba(X_v)[:, 1]
    best_t = thresholds[np.argmax([balanced_accuracy_score(y_v, proba_v >= t) for t in thresholds])]

    lr_models[drug] = (lr, best_t)
    lr_results[drug] = {"A-val_BalAcc": balanced_accuracy_score(y_v, proba_v >= best_t),
                        "A-val_AUC": roc_auc_score(y_v, proba_v)}

    for st, (X_st, Y_st) in test_pp.items():
        ts_mask = ~np.isnan(Y_st[:, di])
        if ts_mask.sum() < 2: continue
        X_ts, y_ts = X_st[ts_mask], Y_st[ts_mask, di].astype(int)
        proba = lr.predict_proba(X_ts)[:, 1]
        preds = proba >= best_t
        lr_results[drug][f"{st}_BalAcc"] = balanced_accuracy_score(y_ts, preds)
        lr_results[drug][f"{st}_AUC"] = roc_auc_score(y_ts, proba)

print("\nLR baseline done.")

In [ ]:
# ── LR Heatmap ──
# Per-drug LR Balanced Accuracy across A-val / B / C / D / B+C+D.
sites_order = ["A-val_BalAcc", "DRIAMS-B_BalAcc", "DRIAMS-C_BalAcc", "DRIAMS-D_BalAcc", "B+C+D_BalAcc"]
data = {}
for drug in DRUGS_10:
    row = [lr_results[drug].get(k, np.nan) for k in sites_order]
    data[drug] = row

fig, ax = plt.subplots(figsize=(10, 8))
hm = pd.DataFrame(data, index=[s.replace("_BalAcc", "") for s in sites_order]).T
sns.heatmap(hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.5, vmax=0.9,
            linewidths=0.5, cbar_kws={"label": "Balanced Accuracy"}, ax=ax)
ax.set_title("LR OneVsRest -- Cross-Site (Train A, Test B/C/D)")
ax.set_xlabel("Test Set"); ax.set_ylabel("Drug")
plt.tight_layout()
plt.savefig(OUT_DIR / "lr_multilabel_heatmap.pdf", bbox_inches="tight")
plt.show()

print("\nLR Per-Drug BalAcc:")
print(hm.to_string())

---
## Shared MLP: Grid Search + Training

6×6 grid scored on validation, then a longer retrain with the weighted loss.

In [ ]:
# =============================================================================
# 5. SHARED MLP -- GRID SEARCH (lr x dropout)
# =============================================================================
# 6x6 lr x dropout grid with the class-weighted loss; candidates are scored
# by per-drug thresholded macro Balanced Accuracy on A validation.

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

LR_GRID = np.linspace(1e-4, 5e-4, 6)
DROP_GRID = np.linspace(0.2, 0.6, 6)
print(f"Grid: {len(LR_GRID)} lr x {len(DROP_GRID)} dropout = {len(LR_GRID)*len(DROP_GRID)} combos")

best_balacc = -1
best_lr = None; best_dh = None

for lr in LR_GRID:
    for d in DROP_GRID:
        dh, dl = d, d / 2
        mlp = MultiLabelMaldiMLP(
            hidden_dim=512, head_dims=(256, 128), use_attention=False,
            dropout_high=dh, dropout_low=dl,
            learning_rate=lr, weight_decay=1e-3,
            batch_size=64, epochs=50, early_stopping_patience=10,
            warmup_epochs=10, val_fraction=0.1,
            random_state=SEED, verbose=False, device=device)
        mlp.fit(X_train_pp, Y_train)

        proba_v = mlp.predict_proba(X_val_pp)
        # Score each candidate with per-drug thresholds tuned on A val (Balanced
        # Accuracy), matching the evaluation protocol used for the MLP runs.
        per_drug_ba = []
        for di in range(10):
            vm = ~np.isnan(Y_val[:, di])
            if vm.sum() < 2: continue
            pv = proba_v[vm, di]; yv = Y_val[vm, di].astype(int)
            bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
            per_drug_ba.append(balanced_accuracy_score(yv, pv >= bt))

        macro_ba = np.mean(per_drug_ba) if per_drug_ba else 0.0
        mark = " *" if macro_ba > best_balacc else ""
        print(f"  lr={lr:.1e}  drop=({dh:.2f},{dl:.2f})  macro_BalAcc={macro_ba:.4f}{mark}")
        if macro_ba > best_balacc:
            best_balacc = macro_ba; best_lr = lr; best_dh = dh

print(f"\nBest: lr={best_lr:.1e} drop=({best_dh:.2f})  macro_BalAcc={best_balacc:.4f}")

In [ ]:
# =============================================================================
# 6. RETRAIN BEST MLP + THRESHOLD TUNING
# =============================================================================
# Retrains the winning configuration for the full schedule, tunes per-drug
# thresholds on A validation and evaluates every test set once; the exported
# model is the best-validation-loss checkpoint.

mlp_best = MultiLabelMaldiMLP(
    hidden_dim=512, head_dims=(256, 128), use_attention=False,
    dropout_high=best_dh, dropout_low=best_dh/2,
    learning_rate=best_lr, weight_decay=1e-4,
    batch_size=64, epochs=100, early_stopping_patience=15,
    warmup_epochs=10, val_fraction=0.1,
    random_state=SEED, verbose=True, device=device)
mlp_best.fit(X_train_pp, Y_train)

# Thresholds tuned once on A validation, then frozen for all test sites.
proba_val = mlp_best.predict_proba(X_val_pp)
mlp_thresholds = []
for di in range(10):
    vm = ~np.isnan(Y_val[:, di])
    if vm.sum() < 2:
        mlp_thresholds.append(0.5)
        continue
    pv = proba_val[vm, di]; yv = Y_val[vm, di].astype(int)
    bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
    mlp_thresholds.append(bt)
mlp_thresholds = np.array(mlp_thresholds)
print(f"\nPer-drug thresholds: {[f'{t:.2f}' for t in mlp_thresholds]}")

# BalAcc/AUC per site, plus the pooled B+C+D row.
mlp_results = {}
for st, (X_st, Y_st) in [("A-val", (X_val_pp, Y_val))] + list(test_pp.items()):
    proba = mlp_best.predict_proba(X_st)
    mlp_results[st] = {}
    for di, drug in enumerate(DRUGS_10):
        tm = ~np.isnan(Y_st[:, di])
        if tm.sum() < 2:
            mlp_results[st][drug] = np.nan
            continue
        preds = (proba[tm, di] >= mlp_thresholds[di])
        yt = Y_st[tm, di].astype(int)
        mlp_results[st][drug] = balanced_accuracy_score(yt, preds)

print("\nMLP done.")

In [ ]:
# ── MLP Heatmap ──
# Same layout for the weighted MLP across all targets.
mlp_data = {}
for drug in DRUGS_10:
    row = [mlp_results[st].get(drug, np.nan) for st in ["A-val", "DRIAMS-B", "DRIAMS-C", "DRIAMS-D", "B+C+D"]]
    mlp_data[drug] = row

fig, ax = plt.subplots(figsize=(10, 8))
hm_mlp = pd.DataFrame(mlp_data, index=["A-val", "B", "C", "D", "B+C+D"]).T
sns.heatmap(hm_mlp, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.5, vmax=0.9,
            linewidths=0.5, cbar_kws={"label": "Balanced Accuracy"}, ax=ax)
ax.set_title("Shared MLP -- Cross-Site (Train A, Test B/C/D)")
ax.set_xlabel("Test Set"); ax.set_ylabel("Drug")
plt.tight_layout()
plt.savefig(OUT_DIR / "mlp_multilabel_heatmap.pdf", bbox_inches="tight")
plt.show()

print("\nMLP Per-Drug BalAcc:")
print(hm_mlp.to_string())

---
## LR vs MLP Comparison

Per-drug summary for the cross-site setting (weighted MLP vs linear baseline).

In [ ]:
# =============================================================================
# 7. LR vs MLP COMPARISON
# =============================================================================
# Joins the LR baseline and the weighted MLP into the per-drug comparison
# table/figure exported to results_multilabel/.

rows = []
for drug in DRUGS_10:
    lr_ba = lr_results[drug].get("B+C+D_BalAcc", np.nan)
    mlp_ba = mlp_results["B+C+D"].get(drug, np.nan)
    lr_av = lr_results[drug].get("A-val_BalAcc", np.nan)
    mlp_av = mlp_results["A-val"].get(drug, np.nan)
    rows.append({"Drug": drug, "LR_BalAcc": lr_ba, "MLP_BalAcc": mlp_ba,
                 "LR_Aval": lr_av, "MLP_Aval": mlp_av})

df_comp = pd.DataFrame(rows)
df_comp_display = df_comp.copy()
df_comp_display["Drug"] = df_comp_display["Drug"].str.slice(0, 15)
print(df_comp_display.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_comp)); w = 0.2
ax.bar(x - w, df_comp["LR_BalAcc"], w, label="LR (OneVsRest)", color="#1f77b4")
ax.bar(x,      df_comp["MLP_BalAcc"], w, label="MLP (Shared)", color="#ff7f0e")
ax.bar(x + w,  df_comp["LR_Aval"], w, label="LR A-val", color="#aec7e8", alpha=0.7)
ax.bar(x + 2*w, df_comp["MLP_Aval"], w, label="MLP A-val", color="#ffbb78", alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([d[:15] for d in df_comp["Drug"]], fontsize=8, rotation=45, ha="right")
ax.set_ylabel("Balanced Accuracy (B+C+D)"); ax.set_title("LR vs Shared MLP -- Cross-Site Multi-Label")
ax.legend(fontsize=8); ax.set_ylim(0, 1); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
plt.grid(True, ls='--', lw=0.5, color='gray', alpha=0.7)
plt.tight_layout(); plt.savefig(OUT_DIR / "multilabel_lr_vs_mlp.pdf")
plt.show()

In [ ]:
# Wrap-up: print the output directory and list the saved PDF figures.
# (Outputs are stripped in the committed notebook; rerun to regenerate.)
print("\nDone. Multi-label results saved to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*.pdf")):
    print(f"  {f.name}")

## Aggregated Shared MLP (A+B+C+D → species-stratified 70/15/15)  [+ class weighting]

Same protocol as 04-00's aggregated run, with the weighted loss; the cache
used here is the legacy 26,642-sample union (see the header note).


In [ ]:
# =============================================================================
# 8. AGGREGATED SHARED MLP (all 4 sites A+B+C+D -> species-stratified split)
# =============================================================================
# Re-runs the full pipeline on the pooled four sites: load union (shared
# cache), species-stratified 70/15/15 split, grid search and final retrain
# with per-drug thresholds on the aggregated validation split.

cache_path = DATA_ROOT / "aggregated_multilabel_data.npz"

if cache_path.exists():
    print(f"Loading aggregated data from cache: {cache_path}")
    cached = np.load(cache_path, allow_pickle=True)
    X_all = cached['X_all']
    Y_all = cached['Y_all']
    sp_all = cached['sp_all']
else:
    print("Loading aggregated data from all 4 sites (first time)...")
    X_train_raw, Y_train_raw, sp_train_raw, test_data_raw = load_multilabel_split()

    # Pool all four sites into one matrix (A+B+C+D) for the aggregated protocol.
    X_all = np.concatenate([X_train_raw] + [v[0] for v in test_data_raw.values()])
    Y_all = np.concatenate([Y_train_raw] + [v[1] for v in test_data_raw.values()])
    sp_all = np.concatenate([sp_train_raw] + [v[2] for v in test_data_raw.values()])

    np.savez_compressed(cache_path, X_all=X_all, Y_all=Y_all, sp_all=sp_all)
    print(f"Data saved to cache: {cache_path}")

print(f"Total aggregated: {X_all.shape[0]} samples, {len(np.unique(sp_all))} species")

# Species-stratified 70/15/15: no species crosses a partition boundary.
idx_trval, idx_test, _, _ = stratified_species_drug_split(
    np.arange(len(Y_all)).reshape(-1, 1),
    np.zeros(len(Y_all)),  # dummy labels
    species=sp_all,
    test_size=0.15,
    random_state=SEED)
idx_trval = idx_trval.flatten().astype(int)
idx_test = idx_test.flatten().astype(int)

X_trval, Y_trval, sp_trval = X_all[idx_trval], Y_all[idx_trval], sp_all[idx_trval]
X_test, Y_test = X_all[idx_test], Y_all[idx_test]

val_frac = 0.15 / 0.85
idx_train, idx_val, _, _ = stratified_species_drug_split(
    np.arange(len(Y_trval)).reshape(-1, 1),
    np.zeros(len(Y_trval)),
    species=sp_trval,
    test_size=val_frac,
    random_state=SEED)
idx_train = idx_train.flatten().astype(int); idx_val = idx_val.flatten().astype(int)
X_train, Y_train = X_trval[idx_train], Y_trval[idx_train]
X_val, Y_val = X_trval[idx_val], Y_trval[idx_val]

print(f"Split: train={X_train.shape[0]}  val={X_val.shape[0]}  test={X_test.shape[0]}")

# Transform fitted on the aggregated training split only.
state_agg = fit_input_transform(X_train, "log1p+standardize")
X_train_pp = apply_input_transform(X_train, state_agg)
X_val_pp   = apply_input_transform(X_val, state_agg)
X_test_pp  = apply_input_transform(X_test, state_agg)
print("Preprocessing done.")

# ── Grid search (6x6 lr x dropout, 36 combos) ──
# Scored on the aggregated validation split with per-drug thresholds.
print(f"\nGrid search: {len(LR_GRID)} lr x {len(DROP_GRID)} dropout = {len(LR_GRID)*len(DROP_GRID)} combos")
agg_best_balacc = -1; agg_best_lr = None; agg_best_dh = None

for lr in LR_GRID:
    for d in DROP_GRID:
        dh, dl = d, d / 2
        mlp = MultiLabelMaldiMLP(
            hidden_dim=512, head_dims=(256, 128), use_attention=False,
            dropout_high=dh, dropout_low=dl,
            learning_rate=lr, weight_decay=1e-3,
            batch_size=64, epochs=50, early_stopping_patience=10,
            warmup_epochs=10, val_fraction=0.1,
            random_state=SEED, verbose=False, device=device)
        mlp.fit(X_train_pp, Y_train)

        proba_v = mlp.predict_proba(X_val_pp)
        per_drug_ba = []
        for di in range(10):
            vm = ~np.isnan(Y_val[:, di])
            if vm.sum() < 2: continue
            pv = proba_v[vm, di]; yv = Y_val[vm, di].astype(int)
            bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
            per_drug_ba.append(balanced_accuracy_score(yv, pv >= bt))
        macro_ba = np.mean(per_drug_ba) if per_drug_ba else 0.0
        mark = " *" if macro_ba > agg_best_balacc else ""
        print(f"  lr={lr:.1e}  drop=({dh:.2f},{dl:.2f})  macro_BalAcc={macro_ba:.4f}{mark}")
        if macro_ba > agg_best_balacc:
            agg_best_balacc = macro_ba; agg_best_lr = lr; agg_best_dh = dh

print(f"\nBest: lr={agg_best_lr:.1e} drop={agg_best_dh:.2f}  macro_BalAcc={agg_best_balacc:.4f}")

# ── Retrain best MLP + threshold tuning ──
# Longer retrain; thresholds tuned on validation and frozen for test.
mlp_agg = MultiLabelMaldiMLP(
    hidden_dim=512, head_dims=(256, 128), use_attention=False,
    dropout_high=agg_best_dh, dropout_low=agg_best_dh/2,
    learning_rate=agg_best_lr, weight_decay=1e-4,
    batch_size=64, epochs=100, early_stopping_patience=15,
    warmup_epochs=10, val_fraction=0.1,
    random_state=SEED, verbose=True, device=device)
mlp_agg.fit(X_train_pp, Y_train)

# Per-drug thresholds
proba_val_agg = mlp_agg.predict_proba(X_val_pp)
agg_thresholds = []
for di in range(10):
    vm = ~np.isnan(Y_val[:, di])
    if vm.sum() < 2:
        agg_thresholds.append(0.5); continue
    pv = proba_val_agg[vm, di]; yv = Y_val[vm, di].astype(int)
    bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
    agg_thresholds.append(bt)
agg_thresholds = np.array(agg_thresholds)

# Evaluate on test set
proba_test = mlp_agg.predict_proba(X_test_pp)
agg_results = {}
for di, drug in enumerate(DRUGS_10):
    tm = ~np.isnan(Y_test[:, di])
    if tm.sum() < 2:
        agg_results[drug] = np.nan; continue
    preds = (proba_test[tm, di] >= agg_thresholds[di])
    yt = Y_test[tm, di].astype(int)
    agg_results[drug] = balanced_accuracy_score(yt, preds)

print("\nAggregated Shared MLP -- Test BalAcc:")
for drug, ba in agg_results.items():
    print(f"  {drug:35s}  {ba:.4f}")

# ── Aggregated vs Cross-Site comparison ──
# Per-drug domain-shift gap: pooled (70/15/15) vs cross-site test scores.
comp_rows = []
for drug in DRUGS_10:
    cs_ba = mlp_results["B+C+D"].get(drug, np.nan)
    ag_ba = agg_results.get(drug, np.nan)
    comp_rows.append({"Drug": drug, "Cross-Site (A->B+C+D)": cs_ba, "Aggregated (70/15/15)": ag_ba})
df_comp2 = pd.DataFrame(comp_rows)
df_comp2["Drug"] = df_comp2["Drug"].str.slice(0, 15)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_comp2)); w = 0.35
ax.bar(x - w/2, df_comp2["Cross-Site (A->B+C+D)"], w, label="Cross-Site A->B+C+D", color="#1f77b4")
ax.bar(x + w/2, df_comp2["Aggregated (70/15/15)"], w, label="Aggregated 70/15/15", color="#ff7f0e")
ax.set_xticks(x); ax.set_xticklabels(df_comp2["Drug"], fontsize=8, rotation=45, ha="right")
ax.set_ylabel("Test BalAcc"); ax.set_title("Shared MLP -- Cross-Site vs Aggregated")
ax.legend(); ax.set_ylim(0, 1); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
plt.grid(True, ls='--', lw=0.5, color='gray', alpha=0.7)
plt.tight_layout(); plt.savefig(OUT_DIR / "multilabel_cross_vs_agg.pdf")
plt.show()

print("\nCross-Site vs Aggregated comparison:")
print(df_comp2.to_string(index=False))